In [ ]:
!pip install torch torchvision torchaudio
!pip install torch-geometric
!pip install pandas numpy scikit-learn networkx

In [ ]:
!pip install torch-geometric -f https://data.pyg.org/whl/torch-2.2.0+cpu.html

Looking in links: https://data.pyg.org/whl/torch-2.2.0+cpu.html


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving cora.cites to cora (1).cites


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving cora.content to cora (1).content


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.transforms import RandomLinkSplit
from sklearn.metrics import average_precision_score, roc_auc_score

In [ ]:
content = np.genfromtxt("cora.content", dtype=str)

node_ids = content[:, 0]
features = content[:, 1:-1].astype(float)

id_map = {nid: i for i, nid in enumerate(node_ids)}
x = torch.tensor(features, dtype=torch.float)

In [ ]:
edges = []
with open("cora.cites") as f:
    for line in f:
        src, dst = line.strip().split()
        edges.append([id_map[src], id_map[dst]])

edge_index = torch.tensor(edges, dtype=torch.long).t()

In [ ]:
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

In [ ]:
data = Data(x=x, edge_index=edge_index)

In [ ]:
split = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=True
)

train_data, val_data, test_data = split(data)

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim, 64)
        self.conv2 = GCNConv(64, 64)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

In [ ]:
def decode(z, edge_index):
    return (z[edge_index[0]] * z[edge_index[1]]).sum(dim=1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GCN(x.size(1)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

train_data = train_data.to(device)
val_data = val_data.to(device)

best_val_ap = 0
patience = 30
wait = 0

for epoch in range(1, 401):
    model.train()
    optimizer.zero_grad()

    z = model(train_data.x, train_data.edge_index)
    pred = decode(z, train_data.edge_label_index)

    loss = F.binary_cross_entropy_with_logits(
        pred, train_data.edge_label.float()
    )
    loss.backward()
    optimizer.step()

    # ---- validation AP ----
    model.eval()
    with torch.no_grad():
        z_val = model(train_data.x, train_data.edge_index)
        val_pred = decode(z_val, val_data.edge_label_index)
        val_ap = average_precision_score(
            val_data.edge_label.cpu(),
            val_pred.cpu()
        )

    if val_ap > best_val_ap:
        best_val_ap = val_ap
        best_state = model.state_dict()
        wait = 0
    else:
        wait += 1
        if wait == patience:
            break

In [ ]:
model.load_state_dict(best_state)
model.eval()

with torch.no_grad():
    z = model(train_data.x, train_data.edge_index)

    test_pred = decode(z, test_data.edge_label_index)

    labels = test_data.edge_label.cpu().numpy()
    scores = test_pred.cpu().numpy()

    ap = average_precision_score(labels, scores)
    auc = roc_auc_score(labels, scores)

print("GCN Test AP :", round(ap,4))
print("GCN Test AUC:", round(auc,4))

GCN Test AP : 0.8767
GCN Test AUC: 0.8695
